# DDP Initial Call

Loads `.env.ddp`, builds the initial IDVerse transaction payload, and posts it to `DDP_API_URL`.

In [3]:
# ============================================================
# CONFIGURATION - run this cell first
# ============================================================
import json
import os
import uuid
from pathlib import Path

import requests
from dotenv import load_dotenv

ENV_FILE = ".env.ddp"

def set_env_value(path, key, value):
    env_path = Path(path)
    lines = env_path.read_text().splitlines() if env_path.exists() else []
    updated = False
    for index, line in enumerate(lines):
        if line.startswith(f"{key}="):
            lines[index] = f"{key}={value}"
            updated = True
            break
    if not updated:
        lines.append(f"{key}={value}")
    env_path.write_text("\n".join(lines) + "\n")

if not Path(ENV_FILE).exists():
    print(f"WARNING: {ENV_FILE} not found - create it with DDP_API_URL and API_KEY")
else:
    load_dotenv(ENV_FILE, override=True)
    print(f"Loaded {ENV_FILE}")

DDP_API_URL = os.getenv("DDP_API_URL", "")
API_KEY = os.getenv("API_KEY", "")

DDP_SESSION_ID = uuid.uuid4().hex
set_env_value(ENV_FILE, "DDP_SESSION_ID", DDP_SESSION_ID)
print(f"Saved generated DDP_SESSION_ID to {ENV_FILE}")
DDP_POLICY = os.getenv("DDP_POLICY") or "Initiate IDVerse"
DDP_SERVICE_TYPE = os.getenv("DDP_SERVICE_TYPE") or "all"
DDP_OUTPUT_FORMAT = os.getenv("DDP_OUTPUT_FORMAT") or "json"

# Placeholder request data - edit before sending a real transaction.
ACCOUNT_FIRST_NAME = os.getenv("DDP_ACCOUNT_FIRST_NAME") or "Jane"
ACCOUNT_LAST_NAME = os.getenv("DDP_ACCOUNT_LAST_NAME") or "Doe"
ACCOUNT_COUNTRY_CODE = os.getenv("DDP_ACCOUNT_COUNTRY_CODE") or "+1"
ACCOUNT_TELEPHONE = os.getenv("DDP_ACCOUNT_TELEPHONE") or "5551234567"
ACCOUNT_EMAIL = os.getenv("DDP_ACCOUNT_EMAIL") or "jane.doe@example.com"

def is_set(value):
    return "set" if value else "NOT SET"

print(f"DDP_API_URL : {DDP_API_URL or 'NOT SET'}")
print(f"API_KEY     : {is_set(API_KEY)}")
print(f"Session ID  : {DDP_SESSION_ID}")
print(f"Policy      : {DDP_POLICY}")
print(f"Service type: {DDP_SERVICE_TYPE}")
print(f"Output fmt  : {DDP_OUTPUT_FORMAT}")
print(f"First name  : {ACCOUNT_FIRST_NAME}")
print(f"Last name   : {ACCOUNT_LAST_NAME}")
print(f"Country code: {ACCOUNT_COUNTRY_CODE}")
print(f"Telephone   : {ACCOUNT_TELEPHONE}")
print(f"Email       : {ACCOUNT_EMAIL}")

Loaded .env.ddp
Saved generated DDP_SESSION_ID to .env.ddp
DDP_API_URL : https://h-api.online-metrix.net/api/session-query
API_KEY     : set
Session ID  : e44bf7c37d2e4cff8fd7525810c6c0e1
Policy      : Initiate IDVerse
Service type: all
Output fmt  : json
First name  : Marco
Last name   : Fanti
Country code: 1
Telephone   : 9419607454
Email       : marco.fanti@lexisnexisrisk.com


---
## Initial Call

In [4]:
# POST to DDP_API_URL - initial call to send link / storeTransaction
if not DDP_API_URL:
    raise ValueError("DDP_API_URL is not set in .env.ddp")
if not API_KEY:
    raise ValueError("API_KEY is not set in .env.ddp")

payload = {
    "org_id": "bmdcdpug",
    "api_key": API_KEY,
    "session_id": DDP_SESSION_ID,
    "policy": DDP_POLICY,
    "service_type": DDP_SERVICE_TYPE,
    "account_first_name": ACCOUNT_FIRST_NAME,
    "account_last_name": ACCOUNT_LAST_NAME,
    "account_country_code": ACCOUNT_COUNTRY_CODE,
    "account_telephone": ACCOUNT_TELEPHONE,
    "event_type": "transaction_other",
    "output_format": DDP_OUTPUT_FORMAT,
    "account_email": ACCOUNT_EMAIL,
}

display_payload = dict(payload)
display_payload["api_key"] = "***" if API_KEY else ""
print("Request payload:")
print(json.dumps(display_payload, indent=2))

resp = requests.post(
    DDP_API_URL,
    json=payload,
    headers={
        "Content-Type": "application/json",
        "accept": "application/json",
    },
    timeout=30,
)

print(f"\nStatus: {resp.status_code}")
try:
    data = resp.json()
    print(json.dumps(data, indent=2))
except ValueError:
    print(resp.text[:1000])
    data = None

resp.raise_for_status()

if not isinstance(data, dict):
    raise ValueError("DDP response did not contain a JSON object")

request_id = data.get("request_id")
if not request_id:
    raise ValueError("DDP response did not include request_id")

set_env_value(ENV_FILE, "DDP_REQUEST_ID", request_id)
print(f"Saved DDP_REQUEST_ID to {ENV_FILE}: {request_id}")

Request payload:
{
  "org_id": "bmdcdpug",
  "api_key": "***",
  "session_id": "e44bf7c37d2e4cff8fd7525810c6c0e1",
  "policy": "Initiate IDVerse",
  "service_type": "all",
  "account_first_name": "Marco",
  "account_last_name": "Fanti",
  "account_country_code": "1",
  "account_telephone": "9419607454",
  "event_type": "transaction_other",
  "output_format": "json",
  "account_email": "marco.fanti@lexisnexisrisk.com"
}

Status: 200
{
  "account_email": "marco.fanti@lexisnexisrisk.com",
  "account_email_activities": [
    "_CHALLENGED"
  ],
  "account_email_attributes": [
    "_CHALLENGED",
    "_CHALLENGE_PASSED"
  ],
  "account_email_contains_dot": "yes",
  "account_email_domain": "lexisnexisrisk.com",
  "account_email_first_seen": "2022-10-20",
  "account_email_last_event": "2026-05-09",
  "account_email_last_update": "2026-05-09",
  "account_email_result": "success",
  "account_email_score": "0",
  "account_email_worst_score": "0",
  "account_first_name": "marco",
  "account_last_na